# Lattes XML to Curriculum Vitae PDF Converter

Converts a Lattes CV export (ZIP containing XML) into a professional, parseable PDF curriculum vitae.

| Block | Responsibility |
|---|---|
| **0** | Dependency installation |
| **1** | ZIP file configuration |
| **2** | XML extraction and parsing |
| **3** | Structured data extraction |
| **4** | PDF generation |


---
## Block 0 - Dependency installation

In [46]:
import subprocess, sys

_DEPS = ['reportlab', 'tqdm', 'ipywidgets']
print('Installing dependencies...')
for _pkg in _DEPS:
    r = subprocess.run(
        [sys.executable, '-m', 'pip', 'install', _pkg, '-q'],
        capture_output=True, text=True
    )
    print(f"  {'OK' if r.returncode == 0 else 'FAILED'}: {_pkg}")
print('\nReady.')

Installing dependencies...
  OK: reportlab
  OK: tqdm
  OK: ipywidgets

Ready.


---
## Block 1 - ZIP file configuration

In [47]:
import os
import pathlib

In [48]:
# EDIT: set the name of the ZIP file exported from the Lattes platform
ZIP_FILE = "CV_6518327334232126.zip"

In [49]:
_DIR      = pathlib.Path(globals().get("__vsc_ipynb_file__", os.getcwd())).parent
_ZIP_PATH = _DIR / ZIP_FILE


In [50]:
if not _ZIP_PATH.exists():
    _zips = sorted(_DIR.glob("*.zip"))
    if _zips:
        _ZIP_PATH = _zips[0]
        ZIP_FILE  = _ZIP_PATH.name
        print(f"ZIP not found by exact name. Using: '{ZIP_FILE}'")
    else:
        raise FileNotFoundError(f"No .zip found in '{_DIR}'.")

PDF_PATH = _ZIP_PATH.with_name(_ZIP_PATH.stem + "_cv.pdf")
print(f"ZIP  : {_ZIP_PATH}")
print(f"PDF  : {PDF_PATH}")
print(f"Size : {_ZIP_PATH.stat().st_size / 1024:.1f} KB")


ZIP  : c:\Users\evers\Downloads\conversor-lattes-para-curriculum-vitae\CV_6518327334232126.zip
PDF  : c:\Users\evers\Downloads\conversor-lattes-para-curriculum-vitae\CV_6518327334232126_cv.pdf
Size : 95.4 KB


---
## Block 2 - XML extraction and parsing

In [51]:
import zipfile
import re
import xml.etree.ElementTree as ET
from tqdm.notebook import tqdm

In [52]:
def extract_xml(zip_path: pathlib.Path) -> ET.Element:
    
    with zipfile.ZipFile(zip_path, 'r') as zf:
        xml_names = [n for n in zf.namelist() if n.lower().endswith('.xml')]
        if not xml_names:
            raise ValueError('No XML file found inside the ZIP archive.')
        with zf.open(xml_names[0]) as fh:
            raw = fh.read()
    try:
        return ET.fromstring(raw)
    except ET.ParseError:

        cleaned = re.sub(rb'encoding=["\'][^"\']+["\']', b'', raw, count=1)
        return ET.fromstring(cleaned)


print('Extracting XML from ZIP...')
with tqdm(total=1, desc='  Extracting', colour='#4CAF50',
          bar_format='{l_bar}{bar:30}{r_bar}') as _pb:
    ROOT = extract_xml(_ZIP_PATH)
    _pb.update(1)

Extracting XML from ZIP...


  Extracting:   0%|                              | 0/1 [00:00<?, ?it/s]

In [53]:
print(f'  Root tag : <{ROOT.tag}>')
print(f"  Lattes ID: {ROOT.get('NUMERO-IDENTIFICADOR', 'N/A')}")
print(f"  Updated  : {ROOT.get('DATA-ATUALIZACAO', 'N/A')}")

  Root tag : <CURRICULO-VITAE>
  Lattes ID: 6518327334232126
  Updated  : 11112025


---
## Block 3 - Structured data extraction

One pure function per section. Returns `dict` or `list[dict]`.

In [54]:
from html import unescape as _unescape

# XML helper utilities

def _attr(elem, key: str, default: str = "") -> str:
    if elem is None:
        return default
    val = elem.get(key, default)
    return _unescape(val).strip() if val else default


def _fmt_date(ddmmyyyy: str) -> str:
    d = ddmmyyyy.strip()
    return f"{d[0:2]}/{d[2:4]}/{d[4:8]}" if len(d) == 8 else d


def _period(yr_s: str, mo_s: str, yr_e: str, mo_e: str) -> str:
    """Build a human-readable period string, e.g. '04/2024 - Present'."""
    start = f"{mo_s}/{yr_s}" if mo_s else yr_s
    end   = (f"{mo_e}/{yr_e}" if mo_e else yr_e) if yr_e else "Present"
    return f"{start} - {end}"


In [55]:
# Section 1: Personal Data

def extract_personal(root: ET.Element) -> dict:
    dg   = root.find("DADOS-GERAIS")
    if dg is None:
        return {}
    res  = dg.find("RESUMO-CV")
    out  = dg.find("OUTRAS-INFORMACOES-RELEVANTES")
    ep   = dg.find(".//ENDERECO-PROFISSIONAL")
    er   = dg.find(".//ENDERECO-RESIDENCIAL")
    email = _attr(ep, "E-MAIL") or _attr(er, "E-MAIL")
    return {
        "name"       : _attr(dg,  "NOME-COMPLETO"),
        "citation"   : _attr(dg,  "NOME-EM-CITACOES-BIBLIOGRAFICAS"),
        "birth"      : _fmt_date(_attr(dg, "DATA-NASCIMENTO")),
        "country"    : _attr(dg,  "PAIS-DE-NASCIMENTO"),
        "orcid"      : _attr(dg,  "ORCID-ID"),
        "email"      : email,
        "homepage"   : _attr(ep,  "HOME-PAGE") or _attr(dg, "HOME-PAGE"),
        "summary"    : _attr(res, "TEXTO-RESUMO-CV-RH")    if res is not None else "",
        "summary_en" : _attr(res, "TEXTO-RESUMO-CV-RH-EN") if res is not None else "",
        "other_info" : _attr(out, "OUTRAS-INFORMACOES-RELEVANTES") if out is not None else "",
    }


In [56]:
# Section 2: Academic Background

_DEGREE_TAGS = [
    ("DOUTORADO",               "NOME-CURSO", "Doctorate"),
    ("MESTRADO",                "NOME-CURSO", "Master's"),
    ("MESTRADO-PROFISSIONAL",   "NOME-CURSO", "Professional Master's"),
    ("ESPECIALIZACAO",          "NOME-CURSO", "Specialization"),
    ("GRADUACAO",               "NOME-CURSO", "Undergraduate"),
    ("APERFEICOAMENTO",         "NOME-CURSO", "Continuing Education / Extension"),
    ("ENSINO-MEDIO-SEGUNDO-GRAU",        "",  "High School"),
    ("ENSINO-FUNDAMENTAL-PRIMEIRO-GRAU", "",  "Primary School"),
]


def extract_education(root: ET.Element) -> list:
    sec = root.find(".//FORMACAO-ACADEMICA-TITULACAO")
    if sec is None:
        return []
    items: list = []
    for tag, course_key, label in _DEGREE_TAGS:
        for el in sec.findall(tag):
            course = _attr(el, course_key) if course_key else ""
            title  = (_attr(el, "TITULO-DA-MONOGRAFIA")
                      or _attr(el, "TITULO-DO-TRABALHO-DE-CONCLUSAO-DE-CURSO"))
            status = _attr(el, "STATUS-DO-CURSO").replace("_", " ").capitalize()
            items.append({
                "type"       : label,
                "course"     : course,
                "institution": _attr(el, "NOME-INSTITUICAO"),
                "start"      : _attr(el, "ANO-DE-INICIO"),
                "end"        : _attr(el, "ANO-DE-CONCLUSAO") or "In progress",
                "status"     : status,
                "title"      : title,
                "workload"   : _attr(el, "CARGA-HORARIA"),
            })
    items.sort(key=lambda x: x["end"], reverse=True)
    return items


In [57]:
# Section 3: Professional Experience

def extract_experience(root: ET.Element) -> list:
    sec = root.find(".//ATUACOES-PROFISSIONAIS")
    if sec is None:
        return []
    items: list   = []
    seen: set     = set()
    for ap in sec.findall("ATUACAO-PROFISSIONAL"):
        inst = _attr(ap, "NOME-INSTITUICAO")
        for vt in ap.findall(".//VINCULOS"):
            role = (_attr(vt, "OUTRO-ENQUADRAMENTO-FUNCIONAL-INFORMADO")
                    or _attr(vt, "OUTRO-VINCULO-INFORMADO")
                    or _attr(vt, "TIPO-DE-VINCULO"))
            yr_s, mo_s = _attr(vt, "ANO-INICIO"), _attr(vt, "MES-INICIO")
            yr_e, mo_e = _attr(vt, "ANO-FIM"),    _attr(vt, "MES-FIM")
            desc = (_attr(vt, "OUTRAS-INFORMACOES-INGLES")
                    or _attr(vt, "OUTRAS-INFORMACOES"))
            key = (inst, role, yr_s)
            if key in seen:
                continue
            seen.add(key)
            items.append({
                "institution": inst,
                "role"       : role,
                "period"     : _period(yr_s, mo_s, yr_e, mo_e),
                "description": desc,
            })
    items.sort(key=lambda x: x["period"], reverse=True)
    return items


In [58]:
# Section 4: Internships

def extract_internships(root: ET.Element) -> list:
    seen:  set  = set()
    items: list = []
    for el in root.findall(".//ESTAGIO"):
        inst     = _attr(el, "NOME-ORGAO")
        activity = _attr(el, "ESTAGIO-REALIZADO")
        key = (inst, activity)
        if key in seen:
            continue
        seen.add(key)
        items.append({
            "institution": inst,
            "activity"   : activity,
            "period"     : _period(
                _attr(el, "ANO-INICIO"), _attr(el, "MES-INICIO"),
                _attr(el, "ANO-FIM"),   _attr(el, "MES-FIM")),
        })
    return items


In [59]:
# Section 5: Research & Development

def extract_research(root: ET.Element) -> list:
    items: list = []
    for pd in root.findall(".//PESQUISA-E-DESENVOLVIMENTO"):
        org  = _attr(pd, "NOME-ORGAO")
        year = _attr(pd, "ANO-INICIO")
        seen_l: set = set()
        lines: list = []
        for ln in pd.findall("LINHA-DE-PESQUISA"):
            t = _attr(ln, "TITULO-DA-LINHA-DE-PESQUISA")
            if t and t not in seen_l:
                seen_l.add(t)
                lines.append(t)
        if org or lines:
            items.append({"org": org, "year": year, "lines": lines})
    return items


In [60]:
# Section 6: Bibliographic Production

def extract_publications(root: ET.Element) -> list:
    sec = root.find(".//PRODUCAO-BIBLIOGRAFICA")
    if sec is None:
        return []
    items: list = []

    def _authors(el) -> str:
        return "; ".join(
            _attr(a, "NOME-COMPLETO-DO-AUTOR") for a in el.findall(".//AUTORES")
        )

    for el in sec.findall(".//ARTIGO-PUBLICADO"):
        d   = el.find("DADOS-BASICOS-DO-ARTIGO")
        if d is None: continue
        det = el.find("DETALHAMENTO-DO-ARTIGO")
        items.append({
            "type"   : "Journal Article",
            "title"  : _attr(d,   "TITULO-DO-ARTIGO"),
            "year"   : _attr(d,   "ANO-DO-ARTIGO"),
            "venue"  : _attr(det, "TITULO-DO-PERIODICO-OU-REVISTA") if det is not None else "",
            "authors": _authors(el),
            "doi"    : _attr(d,   "DOI"),
        })

    for el in sec.findall(".//LIVRO-PUBLICADO-OU-ORGANIZADO"):
        d = el.find("DADOS-BASICOS-DO-LIVRO")
        if d is None: continue
        items.append({
            "type": "Book", "title": _attr(d, "TITULO-DO-LIVRO"),
            "year": _attr(d, "ANO"), "venue": "",
            "authors": _authors(el), "doi": _attr(d, "DOI"),
        })

    for el in sec.findall(".//CAPITULO-DE-LIVRO-PUBLICADO"):
        d = el.find("DADOS-BASICOS-DO-CAPITULO")
        if d is None: continue
        items.append({
            "type": "Book Chapter",
            "title": _attr(d, "TITULO-DO-CAPITULO-DO-LIVRO"),
            "year" : _attr(d, "ANO"), "venue": "",
            "authors": _authors(el), "doi": _attr(d, "DOI"),
        })

    items.sort(key=lambda x: x.get("year", ""), reverse=True)
    return items


In [61]:
# Section 7: Areas of Expertise
# Searches recursively - tag location varies across Lattes XML versions.
# Attribute name fallbacks handle schema variants.

def extract_areas(root: ET.Element) -> list:
    candidates = root.findall(".//AREA-DE-ATUACAO")
    if not candidates:
        return []
    _GRANDE = ["NOME-GRANDE-AREA-DO-CONHECIMENTO", "GRANDE-AREA"]
    _AREA   = ["NOME-DA-AREA-DO-CONHECIMENTO",     "AREA"]
    _ESPEC  = ["NOME-DA-ESPECIALIDADE",             "ESPECIALIDADE"]

    def _first(el, keys: list) -> str:
        for k in keys:
            v = _attr(el, k)
            if v: return v
        return ""

    areas: set = set()
    for el in candidates:
        parts = [p for p in [
            _first(el, _GRANDE),
            _first(el, _AREA),
            _first(el, _ESPEC),
        ] if p]
        if parts:
            areas.add(" > ".join(parts))
    return sorted(areas)


In [62]:
# Section 8: Languages

def extract_languages(root: ET.Element) -> list:
    sec = root.find(".//IDIOMAS")
    if sec is None:
        return []
    return [{
        "language": _attr(el, "IDIOMA"),
        "oral"    : _attr(el, "PROFICIENCIA-DE-CONVERSACAO"),
        "reading" : _attr(el, "PROFICIENCIA-DE-LEITURA"),
        "writing" : _attr(el, "PROFICIENCIA-DE-ESCRITA"),
    } for el in sec.findall("IDIOMA")]


In [63]:
# Section 9: Awards and Titles

def extract_awards(root: ET.Element) -> list:
    sec = root.find(".//PREMIOS-TITULOS")
    if sec is None:
        return []
    items = [{
        "name"  : _attr(el, "NOME-DO-PREMIO-OU-TITULO"),
        "year"  : _attr(el, "ANO-DA-PREMIACAO"),
        "entity": _attr(el, "NOME-DA-ENTIDADE-PROMOTORA"),
    } for el in sec.findall("PREMIO-TITULO")]
    items.sort(key=lambda x: x.get("year", ""), reverse=True)
    return items


In [64]:
# Section 10: Conference & Event Participation

def extract_events(root: ET.Element) -> list:
    items: list = []
    for el in root.findall(".//PARTICIPACAO-EM-CONGRESSO"):
        d   = el.find("DADOS-BASICOS-DA-PARTICIPACAO-EM-CONGRESSO")
        det = el.find("DETALHAMENTO-DA-PARTICIPACAO-EM-CONGRESSO")
        if d is None: continue
        items.append({
            "title"  : _attr(d,   "TITULO"),
            "year"   : _attr(d,   "ANO"),
            "event"  : _attr(det, "NOME-DO-EVENTO") if det is not None else "",
            "country": _attr(d,   "PAIS-DO-EVENTO"),
        })
    items.sort(key=lambda x: x.get("year", ""), reverse=True)
    return items


In [65]:
_PIPELINE = [
    ("Personal data"          , lambda: extract_personal(ROOT)),
    ("Academic background"    , lambda: extract_education(ROOT)),
    ("Professional experience", lambda: extract_experience(ROOT)),
    ("Internships"            , lambda: extract_internships(ROOT)),
    ("Research activities"    , lambda: extract_research(ROOT)),
    ("Publications"           , lambda: extract_publications(ROOT)),
    ("Areas of expertise"     , lambda: extract_areas(ROOT)),
    ("Languages"              , lambda: extract_languages(ROOT)),
    ("Awards"                 , lambda: extract_awards(ROOT)),
    ("Events"                 , lambda: extract_events(ROOT)),
]

print("Extracting data from Lattes XML...")
_DATA: dict = {}
with tqdm(_PIPELINE, desc="  Parsing sections", colour="#FF9800",
          bar_format="{l_bar}{bar:30}{r_bar}") as _pb:
    for _name, _fn in _pb:
        _pb.set_postfix_str(_name)
        _DATA[_name] = _fn()

personal     = _DATA["Personal data"]
education    = _DATA["Academic background"]
experience   = _DATA["Professional experience"]
internships  = _DATA["Internships"]
research     = _DATA["Research activities"]
publications = _DATA["Publications"]
areas        = _DATA["Areas of expertise"]
languages    = _DATA["Languages"]
awards       = _DATA["Awards"]
events       = _DATA["Events"]

print("\nExtraction summary:")
for _k, _v in _DATA.items():
    _n = 1 if isinstance(_v, dict) else len(_v)
    print(f"  {_k:<30} {_n} record(s)")


Extracting data from Lattes XML...


  Parsing sections:   0%|                              | 0/10 [00:00<?, ?it/s]


Extraction summary:
  Personal data                  1 record(s)
  Academic background            9 record(s)
  Professional experience        42 record(s)
  Internships                    6 record(s)
  Research activities            5 record(s)
  Publications                   0 record(s)
  Areas of expertise             0 record(s)
  Languages                      2 record(s)
  Awards                         11 record(s)
  Events                         0 record(s)


---
## Block 4 - PDF generation

**Typography**: Times New Roman (TTF registered from `C:/Windows/Fonts/` for full Unicode support).  
**Layout**: Section titles centered + bold; body text justified; metadata left-aligned smaller text.

### Rich-text encoding strategy
- `_safe(text)` escapes only `& < >` — UTF-8 chars pass through intact to the TTF renderer.
- `_p(text, style)` wraps **plain text** (calls `_safe()` once).
- `_pm(markup, style)` wraps **pre-built markup** (`<b>`, `<i>`) — does NOT call `_safe()` again.
- Bold sections are built as `_pm(f"<b>{_safe(text)}</b>")`, never `_p(f"<b>...")`.

In [66]:
from reportlab.pdfbase import pdfmetrics
from reportlab.pdfbase.ttfonts import TTFont
from reportlab.pdfbase.pdfmetrics import registerFontFamily

In [67]:
_TTF_MAP = {
    'TNR'          : 'C:/Windows/Fonts/times.ttf',
    'TNR-Bold'     : 'C:/Windows/Fonts/timesbd.ttf',
    'TNR-Italic'   : 'C:/Windows/Fonts/timesi.ttf',
    'TNR-BoldItalic': 'C:/Windows/Fonts/timesbi.ttf',
}

FONT_BODY   = 'Times-Roman'
FONT_BOLD   = 'Times-Bold'
FONT_ITALIC = 'Times-Italic'

_n_reg = 0
for _alias, _path in _TTF_MAP.items():
    if os.path.exists(_path):
        pdfmetrics.registerFont(TTFont(_alias, _path))
        _n_reg += 1

if _n_reg >= 2:
    FONT_BODY   = 'TNR'
    FONT_BOLD   = 'TNR-Bold'
    FONT_ITALIC = 'TNR-Italic'
    
    registerFontFamily(
        'TNR',
        normal='TNR',
        bold='TNR-Bold',
        italic='TNR-Italic',
        boldItalic='TNR-BoldItalic',
    )
    print(f'Times New Roman TTF registered ({_n_reg} variants) - full Unicode support active.')
else:
    print('WARNING: Windows TTF not found. Falling back to built-in fonts.')

Times New Roman TTF registered (4 variants) - full Unicode support active.


In [68]:
# Style definitions

from reportlab.lib.pagesizes import A4
from reportlab.lib.units     import cm
from reportlab.lib.enums     import TA_CENTER, TA_LEFT, TA_JUSTIFY
from reportlab.lib           import colors
from reportlab.lib.styles    import ParagraphStyle
from reportlab.platypus      import (
    SimpleDocTemplate, Paragraph, Spacer,
    HRFlowable, Table, TableStyle,
)

_BLUE  = colors.HexColor("#1a3557")
_GRAY  = colors.HexColor("#555555")
_LINE  = colors.HexColor("#b0b8c8")
_STRIP = colors.HexColor("#f4f6fa")

_base = dict(fontName=FONT_BODY, fontSize=10, leading=14, textColor=colors.black)
_bold = dict(fontName=FONT_BOLD, fontSize=10, leading=14, textColor=colors.black)


In [69]:
ST = {
    "name"    : ParagraphStyle("name",    alignment=TA_CENTER, fontSize=20, leading=26,
                               fontName=FONT_BOLD,   textColor=_BLUE),
    "subtitle": ParagraphStyle("subtitle",alignment=TA_CENTER, fontSize=10, leading=14,
                               fontName=FONT_ITALIC,  textColor=_GRAY),
    "meta"    : ParagraphStyle("meta",    alignment=TA_CENTER, fontSize=9,  leading=12,
                               fontName=FONT_BODY,   textColor=_GRAY),
    "section" : ParagraphStyle("section", alignment=TA_CENTER, fontSize=13, leading=18,
                               fontName=FONT_BOLD,   textColor=_BLUE, spaceAfter=2),
    "body"    : ParagraphStyle("body",    alignment=TA_JUSTIFY, **_base,    spaceAfter=2),
    "small"   : ParagraphStyle("small",   alignment=TA_LEFT,   fontSize=9,  leading=12,
                               fontName=FONT_BODY,   textColor=_GRAY),
    "bullet"  : ParagraphStyle("bullet",  alignment=TA_LEFT,   **_base,
                               leftIndent=14, bulletIndent=4, spaceAfter=1),
}
print("Styles defined.")

Styles defined.


In [70]:
# ──────────────────────────────────────────────────────────────────
# Layout primitives
#
# ENCODING STRATEGY (critical for Unicode/accents):
#
#   ReportLab Paragraph parses content as XML. Two rules must hold:
#   1. XML special chars (&, <, >) must be escaped.
#   2. With a TTF font registered, Python UTF-8 strings render natively.
#      Do NOT convert to &#nnn; entities — that can cause double-escape
#      when the same text passes through _safe() more than once.
#
#   _p(text)  : plain text  -> calls _safe() once -> Paragraph
#   _pm(markup): pre-escaped markup (<b>, <i>) -> Paragraph directly
#   Use _pm(f"<b>{_safe(t)}</b>") for bold text. NEVER _p(f"<b>...").
# ──────────────────────────────────────────────────────────────────

In [71]:
def _safe(text: str) -> str:
    """Escape XML meta-characters. UTF-8 chars pass through for TTF rendering."""
    return (
        str(text)
        .replace("&", "&amp;")
        .replace("<", "&lt;")
        .replace(">", "&gt;")
    )


def _p(text: str, style: str = "body") -> Paragraph:
    """Plain-text paragraph. Escapes & < >; UTF-8 chars render via TTF."""
    return Paragraph(_safe(text), ST[style])


def _pm(markup: str, style: str = "body") -> Paragraph:
    """
    Markup paragraph — for pre-assembled strings that already contain
    safe ReportLab XML tags (<b>, <i>, etc.) and pre-escaped text parts.
    Does NOT call _safe() again.
    """
    return Paragraph(markup, ST[style])


def _b(text: str) -> Paragraph:
    """Bullet point with plain text."""
    return Paragraph(f"&#8226; {_safe(text)}", ST["bullet"])


def _hr() -> HRFlowable:
    return HRFlowable(width="100%", thickness=0.8, color=_LINE,
                      spaceAfter=4, spaceBefore=4)


def _sp(h: float = 0.3) -> Spacer:
    return Spacer(1, h * cm)


def _section_header(title: str) -> list:
    """Horizontal rule + centered bold section title + horizontal rule."""
    return [_sp(0.25), _hr(), _p(title.upper(), "section"), _hr(), _sp(0.1)]


In [72]:
# Section builders

def build_header(p: dict) -> list:
    story = [_p(p.get("name", ""), "name")]
    if p.get("citation"):
        story.append(_p(f"Bibliographic citation: {p['citation']}", "subtitle"))
    meta = []
    if p.get("birth"):    meta.append(f"Date of birth: {p['birth']}")
    if p.get("country"): meta.append(f"Country: {p['country']}")
    if p.get("orcid"):   meta.append(f"ORCID: {p['orcid']}")
    if p.get("email"):   meta.append(f"Email: {p['email']}")
    if p.get("homepage"): meta.append(f"Web: {p['homepage']}")
    if meta:
        story.append(_p("  |  ".join(meta), "meta"))
    story.append(_sp(0.4))
    return story


def build_summary(p: dict) -> list:
    text = p.get("summary_en") or p.get("summary", "")
    if not text.strip():
        return []
    return _section_header("Summary") + [_p(text), _sp()]


def build_education(items: list) -> list:
    if not items:
        return []
    story = _section_header("Academic Background")
    for it in items:
        label  = it["type"]
        if it["course"]:
            label += f" in {it['course']}"
        # _pm + _safe: bold markup is pre-built; text is safe
        story.append(_pm(f"<b>{_safe(label)}</b>", "body"))
        period = f"{it['start']} - {it['end']}"
        story.append(_p(f"{it['institution']}  |  {period}  |  {it['status']}", "small"))
        if it.get("title"):
            story.append(_p(f"Thesis/Project: {it['title']}", "small"))
        if it.get("workload"):
            story.append(_p(f"Workload: {it['workload']} h", "small"))
        story.append(_sp(0.2))
    return story


def build_experience(items: list) -> list:
    if not items:
        return []
    story = _section_header("Professional Experience")
    for it in items:
        story.append(_pm(f"<b>{_safe(it['role'])}</b>", "body"))
        story.append(_p(f"{it['institution']}  |  {it['period']}", "small"))
        if it.get("description"):
            desc = it["description"]
            if len(desc) > 700:
                desc = desc[:700] + "..."
            story.append(_p(desc))
        story.append(_sp(0.2))
    return story


def build_internships(items: list) -> list:
    if not items:
        return []
    story = _section_header("Internships")
    for it in items:
        story.append(_b(f"{it['activity']}  -  {it['institution']}  |  {it['period']}"))
    story.append(_sp())
    return story


def build_research(items: list) -> list:
    if not items:
        return []
    story = _section_header("Research and Development Activities")
    for it in items:
        label = it["org"]
        if it.get("year"):
            label += f"  ({it['year']})"
        story.append(_pm(f"<b>{_safe(label)}</b>", "body"))
        for ln in it["lines"][:8]:
            story.append(_b(ln))
        story.append(_sp(0.15))
    return story


def build_publications(items: list) -> list:
    if not items:
        return []
    story = _section_header("Bibliographic Production")
    for it in items:
        parts = [f"[{it['year']}] {it['title']}"]
        if it.get("venue"):   parts.append(it["venue"])
        if it.get("authors"): parts.append(f"Authors: {it['authors']}")
        if it.get("doi"):     parts.append(f"DOI: {it['doi']}")
        story.append(_b("  |  ".join(parts)))
    story.append(_sp())
    return story


def build_areas(items: list) -> list:
    if not items:
        return []
    return _section_header("Areas of Expertise") + [_b(a) for a in items] + [_sp()]


def build_languages(items: list) -> list:
    if not items:
        return []
    story = _section_header("Languages")
    rows = [["Language", "Speaking", "Reading", "Writing"]] + [
        [it["language"], it["oral"], it["reading"], it["writing"]]
        for it in items
    ]
    t = Table(rows, colWidths=[4 * cm, 4 * cm, 4 * cm, 4 * cm])
    t.setStyle(TableStyle([
        ("FONTNAME",       (0, 0), (-1,  0), FONT_BOLD),
        ("FONTNAME",       (0, 1), (-1, -1), FONT_BODY),
        ("FONTSIZE",       (0, 0), (-1, -1), 9),
        ("ALIGN",          (0, 0), (-1, -1), "CENTER"),
        ("GRID",           (0, 0), (-1, -1), 0.5, _LINE),
        ("ROWBACKGROUNDS", (0, 0), (-1, -1), [colors.white, _STRIP]),
        ("TEXTCOLOR",      (0, 0), (-1,  0), _BLUE),
    ]))
    story.append(t)
    story.append(_sp())
    return story


def build_awards(items: list) -> list:
    if not items:
        return []
    story = _section_header("Awards and Titles")
    for it in items:
        line = it["name"]
        if it.get("entity"): line += f"  -  {it['entity']}"
        if it.get("year"):   line += f"  ({it['year']})"
        story.append(_b(line))
    story.append(_sp())
    return story


def build_events(items: list) -> list:
    if not items:
        return []
    story = _section_header("Conference and Event Participation")
    for it in items:
        line = f"[{it.get('year', '?')}] {it.get('title', '')}"
        if it.get("event"):   line += f"  -  {it['event']}"
        if it.get("country"): line += f"  ({it['country']})"
        story.append(_b(line))
    story.append(_sp())
    return story

In [73]:
# Document assembly and PDF generation

def generate_pdf(path: pathlib.Path) -> None:
    """Assemble all sections and write the final PDF to disk."""
    doc = SimpleDocTemplate(
        str(path),
        pagesize=A4,
        topMargin=2.2 * cm,  bottomMargin=2.2 * cm,
        leftMargin=2.5 * cm, rightMargin=2.5 * cm,
        title=f"Curriculum Vitae - {personal.get('name', '')}",
        author="Lattes XML to PDF Converter",
        subject="Academic Curriculum Vitae",
    )
    story: list = []
    story += build_header(personal)
    story += build_summary(personal)
    story += build_areas(areas)
    story += build_education(education)
    story += build_experience(experience)
    story += build_internships(internships)
    story += build_research(research)
    story += build_publications(publications)
    story += build_awards(awards)
    story += build_events(events)
    story += build_languages(languages)
    doc.build(story)


print("Generating PDF...")
with tqdm(total=1, desc="  Building PDF", colour="#9C27B0",
          bar_format="{l_bar}{bar:30}{r_bar}") as _pb:
    generate_pdf(PDF_PATH)
    _pb.update(1)

print(f"\nPDF generated successfully!")
print(f"  File : {PDF_PATH}")
print(f"  Size : {PDF_PATH.stat().st_size / 1024:.1f} KB")


Generating PDF...


  Building PDF:   0%|                              | 0/1 [00:00<?, ?it/s]


PDF generated successfully!
  File : c:\Users\evers\Downloads\conversor-lattes-para-curriculum-vitae\CV_6518327334232126_cv.pdf
  Size : 157.8 KB


---
## Block 5 - Interactive CV Editor

Edit any field inline. Add or remove entries per section. Click **Save & Regenerate PDF** to rebuild.


In [74]:
import copy
import ipywidgets as widgets
from IPython.display import display, clear_output

In [75]:
# Mutable state initialised from extraction results 
_S = {
    'Personal'    : [copy.deepcopy(personal)],
    'Education'   : [copy.deepcopy(e) for e in education],
    'Experience'  : [copy.deepcopy(e) for e in experience],
    'Internships' : [copy.deepcopy(e) for e in internships],
    'Research'    : [copy.deepcopy(e) for e in research],
    'Publications': [copy.deepcopy(e) for e in publications],
    'Areas'       : [{'area': a} for a in areas],
    'Languages'   : [copy.deepcopy(e) for e in languages],
    'Awards'      : [copy.deepcopy(e) for e in awards],
    'Events'      : [copy.deepcopy(e) for e in events],
}

_BLANK = {
    'Education'   : {'type': '', 'course': '', 'institution': '', 'start': '', 'end': '', 'status': '', 'title': '', 'workload': ''},
    'Experience'  : {'institution': '', 'role': '', 'period': '', 'description': ''},
    'Internships' : {'institution': '', 'activity': '', 'period': ''},
    'Research'    : {'org': '', 'year': '', 'lines': []},
    'Publications': {'type': '', 'title': '', 'year': '', 'venue': '', 'authors': '', 'doi': ''},
    'Areas'       : {'area': ''},
    'Languages'   : {'language': '', 'oral': '', 'reading': '', 'writing': ''},
    'Awards'      : {'name': '', 'year': '', 'entity': ''},
    'Events'      : {'title': '', 'year': '', 'event': '', 'country': ''},
}

_LONG = {'summary', 'summary_en', 'description', 'other_info', 'lines'}
_tab_boxes: dict = {}
_status   = widgets.HTML(value='')
_out      = widgets.Output()


def _fw(key: str, value) -> widgets.Widget:
    v = '; '.join(value) if isinstance(value, list) else str(value or '')
    W = widgets.Textarea if (key in _LONG or len(v) > 80) else widgets.Text
    return W(value=v, layout=widgets.Layout(width='96%', height='70px' if W is widgets.Textarea else 'auto'))


def _entry_widget(sec: str, entry: dict) -> widgets.VBox:
    fws = {k: _fw(k, v) for k, v in entry.items()}

    def _sync(change):
        for k, w in fws.items():
            entry[k] = w.value.split('; ') if k == 'lines' else w.value

    for w in fws.values():
        w.observe(_sync, names='value')

    rows = [widgets.HBox([
        widgets.HTML(f"<b style='font-size:12px;min-width:130px;display:inline-block'>{k.replace('_',' ').capitalize()}</b>"),
        fws[k],
    ]) for k in entry]

    del_btn = widgets.Button(description='Remove', button_style='danger',
                             layout=widgets.Layout(width='90px', margin='6px 0'))

    def _del(b):
        if entry in _S[sec]:
            _S[sec].remove(entry)
        _rebuild(sec)

    del_btn.on_click(_del)
    return widgets.VBox(rows + [del_btn],
                         layout=widgets.Layout(border='1px solid #ddd', padding='8px 12px', margin='4px 0'))


def _rebuild(sec: str):
    box      = _tab_boxes[sec]
    entries  = _S[sec]
    children = [_entry_widget(sec, e) for e in entries]

    acc = widgets.Accordion(children=children)
    for idx, e in enumerate(entries):
        title = (e.get('name') or e.get('role') or e.get('type') or e.get('course')
                 or e.get('institution') or e.get('area') or e.get('language')
                 or e.get('title') or e.get('event') or f'Entry {idx + 1}')
        acc.set_title(idx, str(title)[:60])

    add_btn = widgets.Button(description='Add Entry', button_style='info',
                             layout=widgets.Layout(width='110px', margin='8px 0'))

    def _add(b):
        tmpl = _BLANK.get(sec)
        if tmpl:
            _S[sec].append(copy.deepcopy(tmpl))
        elif sec == 'Personal' and not _S[sec]:
            _S[sec].append({})
        _rebuild(sec)

    add_btn.on_click(_add)
    box.children = ([acc, add_btn] if children
                    else [widgets.Label('No entries.'), add_btn])


def _build_editor():
    sections = list(_S.keys())
    tab_children = []
    for sec in sections:
        box = widgets.VBox(layout=widgets.Layout(width='100%', padding='8px'))
        _tab_boxes[sec] = box
        _rebuild(sec)
        tab_children.append(box)

    tab = widgets.Tab(children=tab_children)
    for i, sec in enumerate(sections):
        tab.set_title(i, sec)

    save_btn = widgets.Button(
        description='Save & Regenerate PDF',
        button_style='success',
        layout=widgets.Layout(width='210px'),
    )

    def _save(b):
        with _out:
            clear_output()
            try:
                p   = _S['Personal'][0] if _S['Personal'] else {}
                ar  = [e['area'] for e in _S['Areas'] if e.get('area')]
                story  = []
                story += build_header(p)
                story += build_summary(p)
                story += build_areas(ar)
                story += build_education(_S['Education'])
                story += build_experience(_S['Experience'])
                story += build_internships(_S['Internships'])
                story += build_research(_S['Research'])
                story += build_publications(_S['Publications'])
                story += build_awards(_S['Awards'])
                story += build_events(_S['Events'])
                story += build_languages(_S['Languages'])
                from reportlab.platypus import SimpleDocTemplate
                from reportlab.lib.pagesizes import A4
                from reportlab.lib.units import cm
                doc = SimpleDocTemplate(
                    str(PDF_PATH), pagesize=A4,
                    topMargin=2.2*cm, bottomMargin=2.2*cm,
                    leftMargin=2.5*cm, rightMargin=2.5*cm,
                    title=f"Curriculum Vitae - {p.get('name', '')}",
                    author='Lattes XML to PDF Converter',
                    subject='Academic Curriculum Vitae',
                )
                doc.build(story)
                sz = PDF_PATH.stat().st_size / 1024
                _status.value = f"<span style='color:#2e7d32'>PDF regenerated: {PDF_PATH.name} ({sz:.1f} KB)</span>"
            except Exception as exc:
                _status.value = f"<span style='color:#c62828'>Error: {exc}</span>"

    save_btn.on_click(_save)

    header = widgets.HTML(
        "<h2 style='margin:4px 0'>CV Editor</h2>"
        "<p style='color:#555;margin:0 0 8px'>Edit any field inline. "
        "Add or remove entries. Click <b>Save &amp; Regenerate PDF</b> when done.</p>"
    )
    return widgets.VBox(
        [header, save_btn, _status, tab, _out],
        layout=widgets.Layout(width='100%'),
    )


display(_build_editor())